In [1]:
# Auto-discover CFD datasets and map them to GH surface folders
import re
from pathlib import Path

CFD_DIR = Path("CFDData")
GH_DIR = Path("GHSurfaceData")
GEOM_PREFIX = "gh_nurbs_export"

def normalize_geom_key(text):
    """Normalize geometry names so small naming differences still map correctly."""
    key = re.sub(r"[^a-z0-9]", "", str(text).lower())
    # Legacy spelling fixups
    key = key.replace("monpitch", "monopitch")
    key = key.replace("monpith", "monopitch")
    key = re.sub(r"^mono(\d+d)$", r"monopitch\1", key)
    return key


def normalize_geom_key_legacy(text):
    """Legacy fallback that collapses size variants to a family-level geometry key."""
    key = normalize_geom_key(text)
    key = re.sub(r"(roof)\d+[lwh](?=\d+d$)", r"\1", key)
    key = re.sub(r"\d+[lwh]$", "", key)
    key = re.sub(r"(?<=[a-z])(\d+)$", "", key)
    return key


def extract_geom_from_cfd_name(filename):
    """Extract geometry token from CFDData<geometry>_<wind>d.csv."""
    stem = Path(filename).stem
    stem = re.sub(r"^CFDData", "", stem, flags=re.IGNORECASE)
    stem = re.sub(r"_(-?\d+(?:\.\d+)?)d$", "", stem, flags=re.IGNORECASE)
    return stem

# Build lookup from geometry folders in GHSurfaceData
geom_lookup = {}
geom_lookup_legacy = {}
for folder in sorted([p for p in GH_DIR.iterdir() if p.is_dir()]):
    match = re.search(r"GH_Surface_Dat(?:a)?_(.+)$", folder.name, flags=re.IGNORECASE)
    geom_label = match.group(1) if match else folder.name
    exact_key = normalize_geom_key(geom_label)
    legacy_key = normalize_geom_key_legacy(geom_label)
    geom_lookup[exact_key] = folder.as_posix()
    geom_lookup_legacy.setdefault(legacy_key, folder.as_posix())

datasets = []
unmatched = []
legacy_matches = []
for cfd_path in sorted(CFD_DIR.glob("*.csv")):
    geom_label = extract_geom_from_cfd_name(cfd_path.name)
    geom_key = normalize_geom_key(geom_label)
    geom_folder = geom_lookup.get(geom_key)
    match_mode = "exact"
    if geom_folder is None:
        geom_folder = geom_lookup_legacy.get(normalize_geom_key_legacy(geom_label))
        match_mode = "legacy"
    if geom_folder is None:
        unmatched.append((cfd_path.name, geom_label, geom_key))
        continue
    dataset_name = cfd_path.stem
    datasets.append((dataset_name, cfd_path.as_posix(), geom_folder, GEOM_PREFIX))
    if match_mode != "exact":
        legacy_matches.append((dataset_name, Path(geom_folder).name))

n_total = len(list(CFD_DIR.glob("*.csv")))
print(f"Discovered CFD datasets: {n_total}")
print(f"Mapped datasets: {len(datasets)}")

# Verify mapped datasets have required files
all_files_ok = True
for dataset_name, cfd_file, geom_folder, prefix in datasets:
    cfd_ok = Path(cfd_file).exists()
    geom_ok = Path(geom_folder).exists()
    cp_ok = (Path(geom_folder) / f"{prefix}_control_points.csv").exists()
    knots_ok = (Path(geom_folder) / f"{prefix}_knots.csv").exists()
    surf_ok = (Path(geom_folder) / f"{prefix}_surfaces.csv").exists()
    if cfd_ok and geom_ok and cp_ok and knots_ok and surf_ok:
        print(f"  ✓ {dataset_name} -> {Path(geom_folder).name}")
    else:
        print(f"  ❌ {dataset_name} missing required files in {geom_folder}")
        all_files_ok = False

if unmatched:
    print("\nUnmatched CFD datasets (no geometry folder found):")
    for fname, geom_label, geom_key in unmatched:
        print(f"  - {fname} (geom_token={geom_label!r}, normalised={geom_key!r})")
    available = sorted(geom_lookup.keys())
    print(f"  Available GH keys: {available}")

if legacy_matches:
    print("\nLegacy fallback matches used (review these):")
    for dataset_name, geom_name in legacy_matches:
        print(f"  - {dataset_name} -> {geom_name}")
    print(f"  Legacy matches count: {len(legacy_matches)}")

# HARD ASSERTION: stop the run if anything is unmatched or missing files.
# A silent drop of 3 of 18 files would otherwise be easy to miss in a 75-min compile.
assert len(datasets) == n_total and all_files_ok, (
    f"Discovery problem: {n_total} CFD files but only {len(datasets)} matched. "
    f"Unmatched: {[u[0] for u in unmatched]}. Fix names or add an alias to "
    f"normalize_geom_key before continuing."
)
print(f"\n✓ All {len(datasets)} CFD files mapped successfully")

Discovered CFD datasets: 94
Mapped datasets: 94
  ✓ CFDDataDome05_0d -> GH_Surface_Data_Dome05
  ✓ CFDDataDome10_0d -> GH_Surface_Data_Dome10
  ✓ CFDDataDome16_0d -> GH_Surface_Data_Dome16
  ✓ CFDDataFlatRoof178L_0d -> GH_Surface_Data_FlatRoof178L
  ✓ CFDDataFlatRoof178L_45d -> GH_Surface_Data_FlatRoof178L
  ✓ CFDDataFlatRoof178L_90d -> GH_Surface_Data_FlatRoof178L
  ✓ CFDDataFlatRoof178W_0d -> GH_Surface_Data_FlatRoof178W
  ✓ CFDDataFlatRoof178W_45d -> GH_Surface_Data_FlatRoof178W
  ✓ CFDDataFlatRoof178W_90d -> GH_Surface_Data_FlatRoof178W
  ✓ CFDDataFlatRoof89H_0d -> GH_Surface_Data_FlatRoof89H
  ✓ CFDDataFlatRoof89H_45d -> GH_Surface_Data_FlatRoof89H
  ✓ CFDDataFlatRoof89H_90d -> GH_Surface_Data_FlatRoof89H
  ✓ CFDDataFlatRoof_0d -> GH_Surface_Data_FlatRoof
  ✓ CFDDataFlatRoof_11.25d -> GH_Surface_Data_FlatRoof
  ✓ CFDDataFlatRoof_22.5d -> GH_Surface_Data_FlatRoof
  ✓ CFDDataFlatRoof_45d -> GH_Surface_Data_FlatRoof
  ✓ CFDDataFlatRoof_90d -> GH_Surface_Data_FlatRoof
  ✓ CFDDataGable

In [2]:
# Define core NURBS and feature engineering functions (used by all datasets)
import numpy as np
import pandas as pd
import re
from pathlib import Path

def parse_wind_angle(dataset_title):
    """Extract wind angle deg from filename suffix (e.g. _0d → 0.0)"""
    match = re.search(r"_(-?\d+(?:\.\d+)?)d(?:\.csv)?$", dataset_title)
    if not match:
        raise ValueError(f"Could not parse wind attack angle from: {dataset_title}")
    return float(match.group(1))

def clamped_uniform_knots(n_ctrl, degree, kmin, kmax):
    m = n_ctrl + degree + 1
    n_interior = m - 2 * (degree + 1)
    if n_interior < 0:
        raise ValueError("Invalid NURBS configuration")
    interior = [] if n_interior == 0 else np.linspace(kmin, kmax, n_interior + 2)[1:-1].tolist()
    return np.array(([kmin] * (degree + 1)) + interior + ([kmax] * (degree + 1)), dtype=float)

def bspline_basis_all(u, degree, knots, n_ctrl):
    N = np.zeros((degree + 1, n_ctrl), dtype=float)
    if np.isclose(u, knots[-1]):
        N[0, -1] = 1.0
    else:
        for i in range(n_ctrl):
            if knots[i] <= u < knots[i + 1]:
                N[0, i] = 1.0
    
    for p in range(1, degree + 1):
        for i in range(n_ctrl):
            left = 0.0
            left_den = knots[i + p] - knots[i]
            if left_den > 1e-14:
                left = ((u - knots[i]) / left_den) * N[p - 1, i]
            
            right = 0.0
            if i + 1 < n_ctrl:
                right_den = knots[i + p + 1] - knots[i + 1]
                if right_den > 1e-14:
                    right = ((knots[i + p + 1] - u) / right_den) * N[p - 1, i + 1]
            
            N[p, i] = left + right
    
    return N[degree]

def eval_nurbs_surface(ctrl_xyz, ctrl_w, deg_u, deg_v, knots_u, knots_v, u, v):
    nu, nv, _ = ctrl_xyz.shape
    Nu = bspline_basis_all(u, deg_u, knots_u, nu)
    Nv = bspline_basis_all(v, deg_v, knots_v, nv)
    
    weighted_sum = np.zeros(3, dtype=float)
    weight_sum = 0.0
    for i in range(nu):
        for j in range(nv):
            b = Nu[i] * Nv[j] * ctrl_w[i, j]
            if b != 0.0:
                weighted_sum += b * ctrl_xyz[i, j]
                weight_sum += b
    
    return weighted_sum if abs(weight_sum) < 1e-14 else weighted_sum / weight_sum

def eval_nurbs_normal(ctrl_xyz, ctrl_w, deg_u, deg_v, knots_u, knots_v, u, v, umin, umax, vmin, vmax):
    du = max((umax - umin) * 1e-4, 1e-6)
    dv = max((vmax - vmin) * 1e-4, 1e-6)
    
    u1 = min(max(u + du, umin), umax)
    u0 = min(max(u - du, umin), umax)
    v1 = min(max(v + dv, vmin), vmax)
    v0 = min(max(v - dv, vmin), vmax)
    
    Su1 = eval_nurbs_surface(ctrl_xyz, ctrl_w, deg_u, deg_v, knots_u, knots_v, u1, v)
    Su0 = eval_nurbs_surface(ctrl_xyz, ctrl_w, deg_u, deg_v, knots_u, knots_v, u0, v)
    Sv1 = eval_nurbs_surface(ctrl_xyz, ctrl_w, deg_u, deg_v, knots_u, knots_v, u, v1)
    Sv0 = eval_nurbs_surface(ctrl_xyz, ctrl_w, deg_u, deg_v, knots_u, knots_v, u, v0)
    
    tu = Su1 - Su0
    tv = Sv1 - Sv0
    n = np.cross(tu, tv)
    n_norm = np.linalg.norm(n)
    return np.array([0.0, 0.0, 0.0], dtype=float) if n_norm < 1e-12 else n / n_norm

def farthest_point_subset(points, k):
    n = points.shape[0]
    if n == 0:
        return points
    k = int(max(1, min(k, n)))
    selected = [0]
    min_d2 = np.sum((points - points[0])**2, axis=1)
    for _ in range(1, k):
        idx = int(np.argmax(min_d2))
        selected.append(idx)
        d2 = np.sum((points - points[idx])**2, axis=1)
        min_d2 = np.minimum(min_d2, d2)
    return points[selected]

print("✓ NURBS functions defined")

✓ NURBS functions defined


In [3]:
# Helper functions for flow-path aware features
def compute_leading_edge_distance(cfd_data, surfaces_df, surface_samples, theta, cp_df):
    """
    Compute distance along wind direction from nearest leading edge.
    Returns normalized distance: 0 at leading edge, ~1 at trailing edge per surface.
    """
    wind_vec = np.array([np.cos(theta), -np.sin(theta), 0.0], dtype=float)
    wind_vec = wind_vec / max(np.linalg.norm(wind_vec), 1e-12)

    n_pts = len(cfd_data)
    dist_from_leading_edge = np.full(n_pts, np.nan, dtype=float)

    wind_proj = (cfd_data["x_rel"].to_numpy(dtype=float) * wind_vec[0] +
                 cfd_data["y_rel"].to_numpy(dtype=float) * wind_vec[1] +
                 cfd_data["z_rel"].to_numpy(dtype=float) * wind_vec[2])

    for sid in cfd_data["surface_id"].unique():
        if sid < 0:
            continue
        on_surface = cfd_data["surface_id"].to_numpy() == sid
        if not on_surface.any():
            continue
        on_surface_indices = np.where(on_surface)[0]
        proj = wind_proj[on_surface]
        leading_edge_proj = np.percentile(proj, 5)
        dist = proj - leading_edge_proj
        chord = proj.max() - proj.min()
        if chord > 1e-6:
            dist = dist / chord
        dist_from_leading_edge[on_surface_indices] = dist
    return dist_from_leading_edge


def compute_edge_distance(cfd_data, cp_df, surfaces_df):
    """Signed distance to nearest geometric edge."""
    n_pts = len(cfd_data)
    dist_to_edge = np.full(n_pts, np.nan, dtype=float)
    for sid in surfaces_df["surface_id"].unique():
        surf_cps = cp_df[cp_df["surface_id"] == sid]
        if surf_cps.empty:
            continue
        n_u = int(surfaces_df[surfaces_df["surface_id"] == sid]["cp_count_u"].values[0])
        n_v = int(surfaces_df[surfaces_df["surface_id"] == sid]["cp_count_v"].values[0])
        cps_reshaped = surf_cps.sort_values(["i", "j"])[["x", "y", "z"]].to_numpy(dtype=float).reshape(n_u, n_v, 3)
        edge_pts = np.vstack([
            cps_reshaped[0, :, :], cps_reshaped[-1, :, :],
            cps_reshaped[:, 0, :], cps_reshaped[:, -1, :],
        ])
        on_surface = cfd_data["surface_id"].to_numpy() == sid
        if not on_surface.any():
            continue
        tap_pts = cfd_data.loc[on_surface, ["x_rel", "y_rel", "z_rel"]].to_numpy(dtype=float)
        diff = tap_pts[:, None, :] - edge_pts[None, :, :]
        dist_to_nearest = np.min(np.linalg.norm(diff, axis=2), axis=1)
        interior_pts = cfd_data.loc[on_surface, "surface_dist"].to_numpy() > 0.01
        dist_to_edge[on_surface] = np.where(interior_pts, dist_to_nearest, -dist_to_nearest)
    return dist_to_edge


def compute_streamwise_distance(cfd_data, surface_samples, theta):
    """Streamwise surface distance from stagnation point along wind direction."""
    wind_vec = np.array([np.cos(theta), -np.sin(theta), 0.0], dtype=float)
    wind_vec = wind_vec / max(np.linalg.norm(wind_vec), 1e-12)
    n_pts = len(cfd_data)
    streamwise_dist = np.full(n_pts, np.nan, dtype=float)
    for s in surface_samples:
        sid = int(s["surface_id"])
        pts = s["points"]
        on_surface = cfd_data["surface_id"].to_numpy() == sid
        if not on_surface.any():
            continue
        tap_pts = cfd_data.loc[on_surface, ["x_rel", "y_rel", "z_rel"]].to_numpy(dtype=float)
        proj_on_wind = pts @ wind_vec
        stag_idx = int(np.argmin(proj_on_wind))
        stag_pt = pts[stag_idx]
        vec_from_stag = tap_pts - stag_pt[None, :]
        s_dist = vec_from_stag @ wind_vec
        s_dist = np.maximum(s_dist, 0.0)
        on_surface_indices = np.where(on_surface)[0]
        streamwise_dist[on_surface_indices] = s_dist
    return streamwise_dist


def compute_wind_incidence_angle(cfd_data, theta):
    """
    Local wind incidence angle on surface normal.
    angle = arccos(|dot(wind_dir, normal)|) in radians.
    """
    wind_vec = np.array([np.cos(theta), -np.sin(theta), 0.0], dtype=float)
    wind_vec = wind_vec / max(np.linalg.norm(wind_vec), 1e-12)
    n_wind = np.column_stack([
        cfd_data["nx_rel"].to_numpy(dtype=float),
        cfd_data["ny_rel"].to_numpy(dtype=float),
        cfd_data["nz_rel"].to_numpy(dtype=float),
    ])
    n_mag = np.linalg.norm(n_wind, axis=1, keepdims=True)
    n_wind = np.divide(n_wind, n_mag, out=np.zeros_like(n_wind), where=(n_mag > 1e-12))
    dot_product = np.sum(n_wind * wind_vec[None, :], axis=1)
    dot_product = np.clip(np.abs(dot_product), 0.0, 1.0)
    return np.arccos(dot_product)


def compute_building_metrics(cp_df, theta):
    """
    Compute building bulk descriptors from the geometry control points (NOT tap extents).
    Returns dict with:
      H : building height (z-extent)
      B : crosswind width (in wind-relative frame)
      D : streamwise depth (in wind-relative frame)
      AR_HB, AR_DB : aspect ratios
      frontal_area_proxy : H * B
    All in geometry/CP units.
    """
    pts = cp_df[["x", "y", "z"]].to_numpy(dtype=float)
    # Rotate XY into wind-relative frame using the same convention as for taps
    x_rel = pts[:, 0] * np.cos(theta) + pts[:, 1] * np.sin(theta)
    y_rel = -pts[:, 0] * np.sin(theta) + pts[:, 1] * np.cos(theta)
    z_rel = pts[:, 2]

    H = float(z_rel.max() - z_rel.min())
    B = float(y_rel.max() - y_rel.min())  # crosswind
    D = float(x_rel.max() - x_rel.min())  # streamwise
    H = max(H, 1e-9)
    B = max(B, 1e-9)
    D = max(D, 1e-9)

    return {
        "H": H, "B": B, "D": D,
        "AR_HB": H / B,
        "AR_DB": D / B,
        "frontal_area_proxy": H * B,
        "z_min": float(z_rel.min()),
    }


def compute_z_normalized_H(cfd_data, building_metrics):
    """
    Height above ground normalised by building H (from geometry, not tap cloud).
    """
    z = cfd_data["z_rel"].to_numpy(dtype=float)
    z_min = building_metrics["z_min"]
    H = building_metrics["H"]
    return np.clip((z - z_min) / H, 0.0, 1.5)


def assign_face_category(cfd_data, theta):
    """
    Categorise each tap into a coarse face class based on its wind-relative normal.
    Categories: windward, leeward, side_pos, side_neg, roof, underside.
    Returns a numpy array of strings.
    Wind direction in rotated frame is +x_rel (we rotated coords so that wind blows along +x_rel).
    """
    nx = cfd_data["nx_rel"].to_numpy(dtype=float)
    ny = cfd_data["ny_rel"].to_numpy(dtype=float)
    nz = cfd_data["nz_rel"].to_numpy(dtype=float)
    n = np.stack([nx, ny, nz], axis=1)
    nmag = np.linalg.norm(n, axis=1, keepdims=True)
    n = np.divide(n, nmag, out=np.zeros_like(n), where=(nmag > 1e-12))

    # Vertical-ish: |nz| dominant
    abs_n = np.abs(n)
    dominant = np.argmax(abs_n, axis=1)  # 0=x, 1=y, 2=z

    cat = np.empty(len(n), dtype=object)
    is_z = dominant == 2
    cat[is_z & (n[:, 2] > 0)] = "roof"
    cat[is_z & (n[:, 2] <= 0)] = "underside"

    # Wind blows along +x_rel in the rotated frame (wind_vec = (cos θ, -sin θ, 0) at θ=0 -> +x).
    # Windward face has outward normal pointing UPSTREAM, i.e. nx_rel < 0.
    is_x = (dominant == 0) & ~is_z
    cat[is_x & (n[:, 0] < 0)]  = "windward"
    cat[is_x & (n[:, 0] >= 0)] = "leeward"

    is_y = (dominant == 1) & ~is_z & ~is_x
    cat[is_y & (n[:, 1] > 0)] = "side_pos"
    cat[is_y & (n[:, 1] <= 0)] = "side_neg"

    cat[cat == None] = "other"
    return cat.astype(str)


print("✓ Flow-path aware feature helper functions defined")

✓ Flow-path aware feature helper functions defined


In [4]:
# Neighborhood aggregation functions for spatial dependencies (geometry-only; no Cp leakage)
from scipy.spatial import cKDTree

def compute_neighborhood_features(tap_data, k_neighbors=5):
    """
    For each tap, aggregate GEOMETRY-ONLY statistics from k-nearest neighbours on the same surface.

    Implementation: uses scipy cKDTree (O(N log N) memory) instead of an N×N pairwise distance
    matrix. The previous cdist version was at memory limits on the 15-set faceted pool and would
    OOM on the full pool's Wavy Roof family (surfaces with N > 100k).

    NOTE: must be called with the merged tap-level dataframe `tap_with_surface_ctx`, not raw
    `cfd_data`. The function needs `streamwise_surface_distance` and `curvature_proxy_mean_rad`,
    both added by `build_surface_features` *before* this function is called.

    Returns:
      - neighbor_z_variance: variance of z coordinates in neighbourhood
      - upstream_separation_score: max curvature_proxy among upstream neighbours
      - neighbor_count: number of neighbours actually found per tap
    """
    required_cols = {"streamwise_surface_distance", "curvature_proxy_mean_rad",
                     "surface_id", "x_rel", "y_rel", "z_rel"}
    missing = required_cols - set(tap_data.columns)
    if missing:
        raise ValueError(
            f"compute_neighborhood_features missing required columns: {sorted(missing)}. "
            f"Call this with tap_with_surface_ctx, not raw cfd_data."
        )

    n_pts = len(tap_data)
    neighbor_z_variance = np.full(n_pts, np.nan, dtype=float)
    upstream_separation_score = np.full(n_pts, np.nan, dtype=float)
    neighbor_count = np.full(n_pts, 0, dtype=int)

    surface_id_arr = tap_data["surface_id"].to_numpy()

    for sid in np.unique(surface_id_arr):
        on_surface = surface_id_arr == sid
        on_surface_indices = np.where(on_surface)[0]
        n_on_surface = len(on_surface_indices)
        if n_on_surface < k_neighbors + 1:
            continue

        # Pull per-surface arrays once
        pts_on_surface = tap_data.loc[on_surface, ["x_rel", "y_rel", "z_rel"]].to_numpy(dtype=float)
        z_on_surface = pts_on_surface[:, 2]
        sw_on_surface = tap_data.loc[on_surface, "streamwise_surface_distance"].to_numpy(dtype=float)
        curv_on_surface = tap_data.loc[on_surface, "curvature_proxy_mean_rad"].to_numpy(dtype=float)

        # Build a 2D k-d tree on the (x, y) projection (matches the previous cdist on [:, :2])
        tree = cKDTree(pts_on_surface[:, :2])
        # Query k+1 because the nearest neighbour of any point is itself (distance 0).
        # Drop column 0 of the result (self) to get the k actual neighbours.
        k_query = min(k_neighbors + 1, n_on_surface)
        _, nn_idx = tree.query(pts_on_surface[:, :2], k=k_query)
        nn_idx_neighbours = nn_idx[:, 1:]                       # shape (n_on_surface, k_actual)
        k_actual = nn_idx_neighbours.shape[1]

        for i_local in range(n_on_surface):
            i_global = on_surface_indices[i_local]
            nearest_local = nn_idx_neighbours[i_local]
            neighbor_count[i_global] = int(k_actual)

            # z-variance feature
            neighbor_z_vals = z_on_surface[nearest_local]
            neighbor_z_variance[i_global] = float(np.var(neighbor_z_vals))

            # Upstream-curvature aggregation (R7 fix; was silently NaN before)
            this_streamwise = sw_on_surface[i_local]
            if not np.isnan(this_streamwise):
                neighbour_streamwise = sw_on_surface[nearest_local]
                upstream_mask = neighbour_streamwise < this_streamwise
                if upstream_mask.any():
                    upstream_curvature = curv_on_surface[nearest_local][upstream_mask]
                    if upstream_curvature.size > 0 and not np.all(np.isnan(upstream_curvature)):
                        upstream_separation_score[i_global] = float(np.nanmax(upstream_curvature))

    return neighbor_z_variance, upstream_separation_score, neighbor_count


print("✓ Neighborhood aggregation functions defined (R8b: cKDTree memory fix)")

✓ Neighborhood aggregation functions defined (R8b: cKDTree memory fix)


In [5]:
# Main processing function: load dataset, compute geometry, extract features
def process_dataset(dataset_name, cfd_path, geom_folder, geom_prefix):
    """Process one CFD dataset with its geometry. Returns processed DataFrame."""
    print(f"\n{'='*70}")
    print(f"Processing: {dataset_name}")
    print(f"  CFD: {cfd_path}")
    print(f"  Geom: {geom_folder}")
    print(f"{'='*70}")
    
    # 1) Load CFD dataset
    cfd_data = pd.read_csv(cfd_path)
    n_raw = len(cfd_data)

    # Deduplicate: some CFD exports (notably curved geometries) write each tap twice
    # with identical Cp values. drop_duplicates is safe because we verified the
    # duplicates carry consistent Cp.
    cfd_data = cfd_data.drop_duplicates().reset_index(drop=True)
    n_after = len(cfd_data)

    if n_after < n_raw:
        pct = 100 * (n_raw - n_after) / n_raw
        print(f"✓ Loaded CFD data: {n_raw} rows -> {n_after} after dedup ({pct:.1f}% removed)")
    else:
        print(f"✓ Loaded CFD data: {cfd_data.shape}")
    
    # 2) Parse wind angle from filename
    dataset_title = Path(cfd_path).name
    wind_angle_deg = parse_wind_angle(dataset_title)
    theta = np.deg2rad(wind_angle_deg)
    print(f"  Wind angle: {wind_angle_deg}°")
    
    # 3) Find and rotate coordinates to wind-relative frame
    columns_lower = {col.lower(): col for col in cfd_data.columns}
    x_col = columns_lower.get("x")
    y_col = columns_lower.get("y")
    z_col = columns_lower.get("z")
    if not all([x_col, y_col, z_col]):
        raise KeyError("Missing x, y, z columns in CFD data")
    
    cfd_data["x_rel"] = cfd_data[x_col] * np.cos(theta) + cfd_data[y_col] * np.sin(theta)
    cfd_data["y_rel"] = -cfd_data[x_col] * np.sin(theta) + cfd_data[y_col] * np.cos(theta)
    cfd_data["z_rel"] = cfd_data[z_col]
    print(f"✓ Rotated coordinates")
    
    # 4) Load geometry data
    geom_path = Path(geom_folder)
    surfaces_df = pd.read_csv(geom_path / f"{geom_prefix}_surfaces.csv")
    cp_df = pd.read_csv(geom_path / f"{geom_prefix}_control_points.csv")
    knots_df = pd.read_csv(geom_path / f"{geom_prefix}_knots.csv")
    print(f"✓ Loaded geometry: {len(surfaces_df)} surfaces, {len(cp_df)} control points")
    
    # 5) Compute scale factor and scaled tap positions
    tap_points_raw = cfd_data[[x_col, y_col, z_col]].to_numpy(dtype=float)
    cp_points = cp_df[["x", "y", "z"]].to_numpy(dtype=float)
    
    tap_extent = tap_points_raw.max(axis=0) - tap_points_raw.min(axis=0)
    cp_extent = cp_points.max(axis=0) - cp_points.min(axis=0)
    valid = tap_extent > 1e-12
    scale_candidates = cp_extent[valid] / tap_extent[valid]
    scale_factor = float(np.median(scale_candidates)) if scale_candidates.size else 1.0
    if not np.isfinite(scale_factor) or scale_factor <= 0:
        scale_factor = 1.0
    
    tap_points = tap_points_raw * scale_factor
    print(f"  Scale factor: {scale_factor:.4f}")
    
    # 6) Build surface samples and evaluate normals
    sample_res_u, sample_res_v = 21, 21
    surface_samples = []
    model_centroid = cp_points.mean(axis=0)
    
    for _, surf in surfaces_df.iterrows():
        surface_id = int(surf["surface_id"])
        deg_u, deg_v = int(surf["degree_u"]), int(surf["degree_v"])
        n_u, n_v = int(surf["cp_count_u"]), int(surf["cp_count_v"])
        umin, umax = float(surf["u_min"]), float(surf["u_max"])
        vmin, vmax = float(surf["v_min"]), float(surf["v_max"])
        
        s_cp = cp_df[cp_df["surface_id"] == surface_id].copy().sort_values(["i", "j"])
        if s_cp.shape[0] != n_u * n_v:
            continue
        
        ctrl_xyz = s_cp[["x", "y", "z"]].to_numpy(dtype=float).reshape(n_u, n_v, 3)
        ctrl_w = s_cp["w"].to_numpy(dtype=float).reshape(n_u, n_v) if "w" in s_cp.columns else np.ones((n_u, n_v), dtype=float)
        
        ku_df = knots_df[(knots_df["surface_id"] == surface_id) & (knots_df["dir"].str.upper() == "U")].sort_values("knot_index")
        kv_df = knots_df[(knots_df["surface_id"] == surface_id) & (knots_df["dir"].str.upper() == "V")].sort_values("knot_index")
        
        knots_u = ku_df["knot_value"].to_numpy(dtype=float)
        knots_v = kv_df["knot_value"].to_numpy(dtype=float)
        
        if knots_u.size != n_u + deg_u + 1:
            knots_u = clamped_uniform_knots(n_u, deg_u, umin, umax)
        if knots_v.size != n_v + deg_v + 1:
            knots_v = clamped_uniform_knots(n_v, deg_v, vmin, vmax)
        
        u_vals = np.linspace(umin, umax, sample_res_u)
        v_vals = np.linspace(vmin, vmax, sample_res_v)
        pts, nrm = [], []
        
        for u in u_vals:
            for v in v_vals:
                p = eval_nurbs_surface(ctrl_xyz, ctrl_w, deg_u, deg_v, knots_u, knots_v, u, v)
                n = eval_nurbs_normal(ctrl_xyz, ctrl_w, deg_u, deg_v, knots_u, knots_v, u, v, umin, umax, vmin, vmax)
                if np.linalg.norm(n) > 0 and np.dot(n, p - model_centroid) < 0:
                    n = -n
                pts.append(p)
                nrm.append(n)
        
        surface_samples.append({"surface_id": surface_id, "points": np.array(pts, dtype=float), "normals": np.array(nrm, dtype=float)})
    
    print(f"✓ Built {len(surface_samples)} surface samples")
    
    # 7) Map taps to surfaces and compute normals
    n_pts = tap_points.shape[0]
    best_dist_sq = np.full(n_pts, np.inf, dtype=float)
    best_surface = np.full(n_pts, -1, dtype=int)
    best_normal = np.zeros((n_pts, 3), dtype=float)
    
    for s in surface_samples:
        sp = s["points"]
        sn = s["normals"]
        diff = tap_points[:, None, :] - sp[None, :, :]
        dist_sq = np.einsum("ijk,ijk->ij", diff, diff)
        idx = np.argmin(dist_sq, axis=1)
        dmin = dist_sq[np.arange(n_pts), idx]
        better = dmin < best_dist_sq
        best_dist_sq[better] = dmin[better]
        best_surface[better] = s["surface_id"]
        best_normal[better] = sn[idx[better]]
    
    norm_mag = np.linalg.norm(best_normal, axis=1)
    safe = norm_mag > 1e-12
    best_normal[safe] = best_normal[safe] / norm_mag[safe, None]
    
    cfd_data["surface_id"] = best_surface
    cfd_data["surface_dist"] = np.sqrt(best_dist_sq)
    cfd_data["nx"] = best_normal[:, 0]
    cfd_data["ny"] = best_normal[:, 1]
    cfd_data["nz"] = best_normal[:, 2]
    cfd_data["coord_scale_to_gh"] = scale_factor
    cfd_data["dataset_name"] = dataset_name  # Track source dataset
    print(f"✓ Assigned {len(surface_samples)} surfaces to {len(cfd_data)} taps")
    
    # 8) Rotate normals to wind-relative frame
    cfd_data["nx_rel"] = cfd_data["nx"] * np.cos(theta) + cfd_data["ny"] * np.sin(theta)
    cfd_data["ny_rel"] = -cfd_data["nx"] * np.sin(theta) + cfd_data["ny"] * np.cos(theta)
    cfd_data["nz_rel"] = cfd_data["nz"]
    cfd_data["n_rel_mag"] = np.sqrt(cfd_data["nx_rel"]**2 + cfd_data["ny_rel"]**2 + cfd_data["nz_rel"]**2)
    print(f"✓ Rotated normals to wind-relative frame")
    
    return cfd_data, surfaces_df, cp_df, knots_df, surface_samples, theta

print("✓ Processing function defined")

✓ Processing function defined


In [6]:
# Extract surface features and build feature-rich dataset
def _parse_roof_tilt_from_name(dataset_name):
    """Extract roof tilt in degrees from dataset_name. Examples:
       'CFDDataGableRoof5d_0d'   -> 5
       'CFDDataMonoPitchRoof15d_45d' -> 15
       'CFDDataFlatRoof_22.5d'   -> 0    (FlatRoof has no tilt)
       'CFDDataDome10_0d'        -> 0    (Dome has no roof tilt; the 10 is a scale suffix)
       'CFDDataWaves10_45d'      -> 0
    Returns degrees as a float.
    """
    name = str(dataset_name)
    # Look for a tilt token inside the geometry name: ...Roof<N>d_  (capital-letter-then-digit-d)
    m = re.search(r"Roof(\d+(?:\.\d+)?)d_", name)
    if m:
        return float(m.group(1))
    # Gable<N>d_ (legacy short form, no "Roof" word)
    m = re.search(r"(?:Gable|MonoPitch|Monopitch)(\d+(?:\.\d+)?)d_", name)
    if m:
        return float(m.group(1))
    return 0.0


def build_surface_features(cfd_data, surfaces_df, cp_df, knots_df, surface_samples, theta, dataset_name=""):
    """Compute surface-level descriptors and merge into tap-level features."""

    # 0) Building bulk descriptors from geometry control points
    bldg = compute_building_metrics(cp_df, theta)
    H = bldg["H"]

    # NEW: bulk curvature descriptors — distinguishes faceted (low) from curved (high) geometries
    cp_pts = cp_df[["x", "y", "z"]].to_numpy(dtype=float)
    if len(cp_pts) >= 4:
        # Estimate per-point local curvature proxy from variance of nearest-neighbour offset directions
        from scipy.spatial.distance import cdist as _cdist
        d_cp = _cdist(cp_pts, cp_pts)
        np.fill_diagonal(d_cp, np.inf)
        k_local = min(8, len(cp_pts) - 1)
        local_curv = np.zeros(len(cp_pts), dtype=float)
        for i in range(len(cp_pts)):
            nn_idx = np.argsort(d_cp[i])[:k_local]
            offsets = cp_pts[nn_idx] - cp_pts[i]
            offsets_n = offsets / (np.linalg.norm(offsets, axis=1, keepdims=True) + 1e-12)
            # Variance of unit offsets is high for curved surfaces, low for flat ones
            local_curv[i] = float(np.mean(np.var(offsets_n, axis=0)))
        bldg_max_curvature = float(np.max(local_curv))
        bldg_mean_curvature = float(np.mean(local_curv))
    else:
        bldg_max_curvature = 0.0
        bldg_mean_curvature = 0.0

    # 1) Extract surface features from samples (unchanged from previous patch)
    surface_feature_rows = []
    for s in surface_samples:
        sid = int(s["surface_id"])
        pts = s["points"]
        nrm = s["normals"]
        surf_meta = surfaces_df[surfaces_df["surface_id"] == sid].iloc[0]

        mean_n = nrm.mean(axis=0)
        mean_n_mag = np.linalg.norm(mean_n)
        mean_n = mean_n / mean_n_mag if mean_n_mag > 1e-12 else np.array([0.0, 0.0, 0.0], dtype=float)

        cos_to_mean = np.clip(nrm @ mean_n, -1.0, 1.0)
        angle_to_mean = np.arccos(cos_to_mean)
        curvature_proxy_mean = float(np.mean(angle_to_mean))
        curvature_proxy_max = float(np.max(angle_to_mean))

        centered = pts - pts.mean(axis=0, keepdims=True)
        _, _, vh = np.linalg.svd(centered, full_matrices=False)
        plane_n = vh[-1] / max(np.linalg.norm(vh[-1]), 1e-12)
        signed_dist = centered @ plane_n
        roughness_rms = float(np.sqrt(np.mean(signed_dist**2)))

        bbox_min = pts.min(axis=0); bbox_max = pts.max(axis=0)
        bbox_size = bbox_max - bbox_min
        bbox_diag = float(np.linalg.norm(bbox_size))
        bbox_volume = float(np.prod(np.maximum(bbox_size, 0.0)))

        pca_vals = np.linalg.svd(centered, full_matrices=False, compute_uv=False)
        pca_vals = pca_vals / max(np.sum(pca_vals), 1e-12)
        shape_linearity = float(pca_vals[0] - pca_vals[1]) if pca_vals.size >= 2 else 0.0
        shape_planarity = float(pca_vals[1] - pca_vals[2]) if pca_vals.size >= 3 else 0.0
        shape_scattering = float(pca_vals[2]) if pca_vals.size >= 3 else 0.0

        nz_abs = float(abs(mean_n[2]))
        mean_n_rel_x = float(mean_n[0] * np.cos(theta) + mean_n[1] * np.sin(theta))
        mean_n_rel_y = float(-mean_n[0] * np.sin(theta) + mean_n[1] * np.cos(theta))
        mean_n_rel_z = float(mean_n[2])

        on_surface_mask = cfd_data["surface_id"].to_numpy() == sid
        taps_on_surface = int(on_surface_mask.sum())
        mean_tap_dist = float(cfd_data.loc[on_surface_mask, "surface_dist"].mean()) if taps_on_surface > 0 else 0.0

        surface_feature_rows.append({
            "surface_id": sid,
            "area_from_export": float(surf_meta["area"]),
            "is_trimmed_face": bool(surf_meta["is_trimmed_face"]),
            "loop_count": int(surf_meta["loop_count"]),
            "mean_nx": float(mean_n[0]),
            "mean_ny": float(mean_n[1]),
            "mean_nz": float(mean_n[2]),
            "mean_nx_rel": mean_n_rel_x,
            "mean_ny_rel": mean_n_rel_y,
            "mean_nz_rel": mean_n_rel_z,
            "curvature_proxy_mean_rad": curvature_proxy_mean,
            "curvature_proxy_max_rad": curvature_proxy_max,
            "roughness_rms": roughness_rms,
            "normal_dispersion_std_rad": float(np.std(angle_to_mean)),
            "verticality_abs_nz": nz_abs,
            "bbox_diag": bbox_diag,
            "bbox_volume": bbox_volume,
            "shape_linearity": shape_linearity,
            "shape_planarity": shape_planarity,
            "shape_scattering": shape_scattering,
            "sample_count": int(pts.shape[0]),
            "tap_count": taps_on_surface,
            "tap_surface_dist_mean": mean_tap_dist,
        })

    surface_features_df = pd.DataFrame(surface_feature_rows).sort_values("surface_id").reset_index(drop=True)

    # 2) Pairwise surface context (unchanged)
    surface_centroids_df = cp_df.groupby("surface_id", as_index=False)[["x", "y", "z"]].mean().rename(
        columns={"x": "cx", "y": "cy", "z": "cz"}
    )
    surface_ctx_df = surface_features_df.merge(surface_centroids_df, on="surface_id", how="left")

    sid_arr = surface_ctx_df["surface_id"].to_numpy(dtype=int)
    C = surface_ctx_df[["cx", "cy", "cz"]].to_numpy(dtype=float)
    N = surface_ctx_df[["mean_nx", "mean_ny", "mean_nz"]].to_numpy(dtype=float)
    A = surface_ctx_df["area_from_export"].to_numpy(dtype=float)
    K = surface_ctx_df["curvature_proxy_mean_rad"].to_numpy(dtype=float)
    R = surface_ctx_df["roughness_rms"].to_numpy(dtype=float)

    n_surf = len(surface_ctx_df)
    wind_vec = np.array([np.cos(theta), -np.sin(theta), 0.0], dtype=float)
    wind_vec = wind_vec / max(np.linalg.norm(wind_vec), 1e-12)

    if n_surf >= 2:
        delta = C[:, None, :] - C[None, :, :]
        dist = np.linalg.norm(delta, axis=2)
        dist_safe = np.where(dist > 1e-12, dist, np.inf)
        dotN = np.clip(N @ N.T, -1.0, 1.0)
        W = 1.0 / dist_safe
        np.fill_diagonal(W, 0.0)
        Wsum = W.sum(axis=1)
        Wnorm = np.divide(W, Wsum[:, None], out=np.zeros_like(W), where=(Wsum[:, None] > 0))
        neighbor_curv_wmean = (Wnorm * K[None, :]).sum(axis=1)
        neighbor_rough_wmean = (Wnorm * R[None, :]).sum(axis=1)
        dot_for_opp = dotN.copy()
        np.fill_diagonal(dot_for_opp, 1.0)
        opp_idx = np.argmin(dot_for_opp, axis=1)
        opp_normal_dot = dotN[np.arange(n_surf), opp_idx]
        center_global = C.mean(axis=0)
        proj = (C - center_global) @ wind_vec
        proj_scale = max(np.std(proj), 1e-12)
        wind_pos_norm = proj / proj_scale
        upstream_area_ratio = np.zeros(n_surf, dtype=float)
        for i in range(n_surf):
            upstream_mask = proj < proj[i]
            area_total = max(A.sum(), 1e-12)
            upstream_area_ratio[i] = A[upstream_mask].sum() / area_total
        surface_ctx_df["neighbor_curv_wmean"] = neighbor_curv_wmean
        surface_ctx_df["neighbor_rough_wmean"] = neighbor_rough_wmean
        surface_ctx_df["opp_normal_dot"] = opp_normal_dot
        surface_ctx_df["wind_pos_norm"] = wind_pos_norm
        surface_ctx_df["upstream_area_ratio"] = upstream_area_ratio

    tap_with_surface_ctx = cfd_data.merge(surface_ctx_df, on="surface_id", how="left", validate="m:1")

    # 3) Flow-path aware features
    print(f"  → Computing flow-path aware features...")
    tap_with_surface_ctx["dist_from_leading_edge_along_wind"] = compute_leading_edge_distance(
        cfd_data, surfaces_df, surface_samples, theta, cp_df
    )
    tap_with_surface_ctx["dist_to_nearest_edge"] = compute_edge_distance(cfd_data, cp_df, surfaces_df)
    tap_with_surface_ctx["streamwise_surface_distance"] = compute_streamwise_distance(
        cfd_data, surface_samples, theta
    )
    tap_with_surface_ctx["wind_incidence_angle_rad"] = compute_wind_incidence_angle(cfd_data, theta)
    tap_with_surface_ctx["z_over_H"] = compute_z_normalized_H(cfd_data, bldg)
    tap_with_surface_ctx["dist_from_leading_edge_H"] = (
        tap_with_surface_ctx["streamwise_surface_distance"].to_numpy(dtype=float) / H
    )
    tap_with_surface_ctx["dist_to_nearest_edge_H"] = (
        tap_with_surface_ctx["dist_to_nearest_edge"].to_numpy(dtype=float) / H
    )
    print(f"  ✓ Flow-path features computed")

    # 4) Neighborhood (geometry-only) features — no Cp leakage
    print(f"  → Computing neighborhood aggregation features...")
    neighbor_z_variance, upstream_separation_score, neighbor_count = compute_neighborhood_features(
        tap_with_surface_ctx, k_neighbors=5
    )
    tap_with_surface_ctx["neighbor_z_variance"] = neighbor_z_variance
    tap_with_surface_ctx["upstream_separation_score"] = upstream_separation_score
    tap_with_surface_ctx["neighbor_count"] = neighbor_count
    print(f"  ✓ Neighborhood features computed")

    # 5) Wind-angle scalars (broadcast)
    tap_with_surface_ctx["wind_angle_rad"] = float(theta)
    tap_with_surface_ctx["wind_sin"] = float(np.sin(theta))
    tap_with_surface_ctx["wind_cos"] = float(np.cos(theta))

    # 6) Building bulk descriptors (broadcast) — extended with curvature
    tap_with_surface_ctx["bldg_H"] = bldg["H"]
    tap_with_surface_ctx["bldg_B"] = bldg["B"]
    tap_with_surface_ctx["bldg_D"] = bldg["D"]
    tap_with_surface_ctx["bldg_AR_HB"] = bldg["AR_HB"]
    tap_with_surface_ctx["bldg_AR_DB"] = bldg["AR_DB"]
    tap_with_surface_ctx["bldg_max_curvature"] = bldg_max_curvature   # NEW
    tap_with_surface_ctx["bldg_mean_curvature"] = bldg_mean_curvature # NEW

    # 7) Roof-tilt features (broadcast). Parsed from dataset_name.
    roof_tilt_deg = _parse_roof_tilt_from_name(dataset_name)
    roof_tilt_rad = np.deg2rad(roof_tilt_deg)
    tap_with_surface_ctx["roof_tilt_rad"] = float(roof_tilt_rad)
    tap_with_surface_ctx["roof_tilt_sin"] = float(np.sin(roof_tilt_rad))
    tap_with_surface_ctx["roof_tilt_cos"] = float(np.cos(roof_tilt_rad))

    # 8) Face-category features.
    #    a) Discrete one-hots (works well on faceted geometries; degenerate but harmless on curved).
    face_cat = assign_face_category(cfd_data, theta)
    tap_with_surface_ctx["face_is_windward"]  = (face_cat == "windward").astype(np.float32)
    tap_with_surface_ctx["face_is_leeward"]   = (face_cat == "leeward").astype(np.float32)
    tap_with_surface_ctx["face_is_roof"]      = (face_cat == "roof").astype(np.float32)
    tap_with_surface_ctx["face_is_underside"] = (face_cat == "underside").astype(np.float32)
    tap_with_surface_ctx["face_is_side_pos"]  = (face_cat == "side_pos").astype(np.float32)
    tap_with_surface_ctx["face_is_side_neg"]  = (face_cat == "side_neg").astype(np.float32)

    #    b) NEW continuous orientation: spherical coordinates of the wind-relative normal.
    #       For curved geometries (Dome, Waves) every tap has a unique normal, so this is
    #       where the orientation signal actually lives.
    nx_r = tap_with_surface_ctx["nx_rel"].to_numpy(dtype=float)
    ny_r = tap_with_surface_ctx["ny_rel"].to_numpy(dtype=float)
    nz_r = tap_with_surface_ctx["nz_rel"].to_numpy(dtype=float)
    nrm_mag = np.sqrt(nx_r**2 + ny_r**2 + nz_r**2) + 1e-12
    nx_u, ny_u, nz_u = nx_r / nrm_mag, ny_r / nrm_mag, nz_r / nrm_mag
    # Polar angle from +z axis (0 = roof-up, π = underside, π/2 = vertical-walled)
    tap_with_surface_ctx["face_normal_polar_rad"] = np.arccos(np.clip(nz_u, -1.0, 1.0))
    # Azimuth in the (x_rel, y_rel) plane: 0 = leeward (+x), ±π = windward (-x)
    tap_with_surface_ctx["face_normal_azimuth_rad"] = np.arctan2(ny_u, nx_u)
    # Sin/cos so the azimuth wraps smoothly (avoids the ±π discontinuity)
    tap_with_surface_ctx["face_normal_azimuth_sin"] = np.sin(tap_with_surface_ctx["face_normal_azimuth_rad"])
    tap_with_surface_ctx["face_normal_azimuth_cos"] = np.cos(tap_with_surface_ctx["face_normal_azimuth_rad"])

    # 9) Output column selection
    target_col = "Cp" if "Cp" in tap_with_surface_ctx.columns else None

    lean_context_feature_cols = [
        "surface_id",
        "x_rel", "y_rel", "z_rel",
        "nx_rel", "ny_rel", "nz_rel",
        "curvature_proxy_mean_rad", "roughness_rms", "verticality_abs_nz", "area_from_export",
        "neighbor_curv_wmean", "opp_normal_dot", "wind_pos_norm", "upstream_area_ratio",
        "dist_from_leading_edge_along_wind",
        "dist_to_nearest_edge",
        "streamwise_surface_distance",
        "wind_incidence_angle_rad",
        "z_over_H",
        "dist_from_leading_edge_H",
        "dist_to_nearest_edge_H",
        "neighbor_z_variance", "upstream_separation_score",
        "wind_angle_rad", "wind_sin", "wind_cos",
        "bldg_H", "bldg_B", "bldg_D", "bldg_AR_HB", "bldg_AR_DB",
        "bldg_max_curvature", "bldg_mean_curvature",   # NEW
        "roof_tilt_rad", "roof_tilt_sin", "roof_tilt_cos",  # NEW
        "face_is_windward", "face_is_leeward", "face_is_roof",
        "face_is_underside", "face_is_side_pos", "face_is_side_neg",
        "face_normal_polar_rad",                # NEW
        "face_normal_azimuth_sin", "face_normal_azimuth_cos",  # NEW (wrap-safe)
        "dataset_name",
    ]
    lean_context_feature_cols = [c for c in lean_context_feature_cols if c in tap_with_surface_ctx.columns]

    nn_lean_context_df = tap_with_surface_ctx[lean_context_feature_cols + ([target_col] if target_col else [])].copy()

    essential_cols = [c for c in ["x_rel", "y_rel", "z_rel", "nx_rel", "ny_rel", "nz_rel"] if c in nn_lean_context_df.columns]
    if target_col:
        nn_lean_context_df = nn_lean_context_df.dropna(subset=[target_col] + essential_cols).reset_index(drop=True)
    else:
        nn_lean_context_df = nn_lean_context_df.dropna(subset=essential_cols).reset_index(drop=True)

    optional_pred_cols = [c for c in nn_lean_context_df.columns
                          if c not in ([target_col] if target_col else []) + ["dataset_name"]]
    cols_to_fill = [c for c in optional_pred_cols if nn_lean_context_df[c].isna().any()]
    if cols_to_fill:
        for c in cols_to_fill:
            if pd.api.types.is_numeric_dtype(nn_lean_context_df[c]):
                nn_lean_context_df[c] = nn_lean_context_df[c].fillna(0.0)

    print(f"  ✓ Features: {len(lean_context_feature_cols)} cols, {len(nn_lean_context_df)} rows")
    return nn_lean_context_df


print("✓ Feature extraction function defined (D2: curved-geom support, roof tilt, polar/azimuth normals)")

✓ Feature extraction function defined (D2: curved-geom support, roof tilt, polar/azimuth normals)


In [7]:
# Process all discovered datasets and compile into single output
from pathlib import Path

all_processed_data = []

for dataset_name, cfd_file, geom_folder, geom_prefix in datasets:
    try:
        # Process dataset
        cfd_data, surfaces_df, cp_df, knots_df, surface_samples, theta = process_dataset(
            dataset_name, cfd_file, geom_folder, geom_prefix
        )
        
        # Extract features
        processed_df = build_surface_features(cfd_data, surfaces_df, cp_df, knots_df, surface_samples, theta, dataset_name=dataset_name)
        all_processed_data.append(processed_df)
        
        print(f"✓ {dataset_name}: {len(processed_df)} rows processed\n")
    
    except Exception as e:
        print(f"❌ {dataset_name} failed: {e}\n")

# Compile all datasets
if all_processed_data:
    compiled_df = pd.concat(all_processed_data, ignore_index=True)
    print(f"\n{'='*70}")
    print(f"COMPILATION COMPLETE")
    print(f"{'='*70}")
    print(f"Total rows: {len(compiled_df)}")
    print(f"Columns: {len(compiled_df.columns)}")
    print(f"Datasets included: {compiled_df['dataset_name'].nunique()}")
    print(f"\nDataset breakdown:")
    for name, count in compiled_df['dataset_name'].value_counts().items():
        print(f"  {name}: {count}")
    
    # Save compiled dataset
    output_dir = Path("ProcessedData")
    output_dir.mkdir(exist_ok=True)
    output_csv = output_dir / "compiled_processed_features.csv"
    compiled_df.to_csv(output_csv, index=False)
    print(f"\n✓ Saved to: {output_csv}")
    print(f"  Shape: {compiled_df.shape}")
    print(f"  Columns: {list(compiled_df.columns)}")
else:
    print("❌ No datasets processed successfully.")


Processing: CFDDataDome05_0d
  CFD: CFDData/CFDDataDome05_0d.csv
  Geom: GHSurfaceData/GH_Surface_Data_Dome05
✓ Loaded CFD data: (12541, 4)
  Wind angle: 0.0°
✓ Rotated coordinates
✓ Loaded geometry: 8 surfaces, 84 control points
  Scale factor: 120.7378
✓ Built 8 surface samples
✓ Assigned 8 surfaces to 12541 taps
✓ Rotated normals to wind-relative frame
  → Computing flow-path aware features...
  ✓ Flow-path features computed
  → Computing neighborhood aggregation features...
  ✓ Neighborhood features computed
  ✓ Features: 47 cols, 12541 rows
✓ CFDDataDome05_0d: 12541 rows processed


Processing: CFDDataDome10_0d
  CFD: CFDData/CFDDataDome10_0d.csv
  Geom: GHSurfaceData/GH_Surface_Data_Dome10
✓ Loaded CFD data: (49706, 4)
  Wind angle: 0.0°
✓ Rotated coordinates
✓ Loaded geometry: 8 surfaces, 84 control points
  Scale factor: 120.7177
✓ Built 8 surface samples
✓ Assigned 8 surfaces to 49706 taps
✓ Rotated normals to wind-relative frame
  → Computing flow-path aware features...
  ✓ 

In [8]:
# Verification and summary of compiled dataset
output_csv = Path("ProcessedData") / "compiled_processed_features.csv"

if output_csv.exists():
    final_df = pd.read_csv(output_csv)
    
    print(f"\n{'='*70}")
    print(f"FINAL DATASET SUMMARY")
    print(f"{'='*70}")
    print(f"\nShape: {final_df.shape}")
    print(f"Memory: {final_df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    
    print(f"\nDataset sources:")
    for name in sorted(final_df['dataset_name'].unique()):
        count = (final_df['dataset_name'] == name).sum()
        print(f"  {name}: {count} rows")
    
    print(f"\nFeature columns ({len(final_df.columns)}):")
    for i, col in enumerate(final_df.columns, 1):
        dtype = str(final_df[col].dtype)
        null_pct = 100 * final_df[col].isna().sum() / len(final_df)
        print(f"  {i:2d}. {col:30s} ({dtype:8s}, {null_pct:5.1f}% null)")
    
    print(f"\nTarget variable (Cp) statistics:")
    if 'Cp' in final_df.columns:
        print(f"  Count: {final_df['Cp'].notna().sum()}")
        print(f"  Mean:  {final_df['Cp'].mean():.4f}")
        print(f"  Std:   {final_df['Cp'].std():.4f}")
        print(f"  Min:   {final_df['Cp'].min():.4f}")
        print(f"  Max:   {final_df['Cp'].max():.4f}")
    else:
        print("  ❌ Cp column not found")
    
    print(f"\n✓ Dataset ready for ML training")
else:
    print(f"❌ Output file not found: {output_csv}")


FINAL DATASET SUMMARY

Shape: (5705217, 48)
Memory: 2602.63 MB

Dataset sources:
  CFDDataDome05_0d: 12541 rows
  CFDDataDome10_0d: 49706 rows
  CFDDataDome16_0d: 105526 rows
  CFDDataFlatRoof178L_0d: 57219 rows
  CFDDataFlatRoof178L_45d: 57148 rows
  CFDDataFlatRoof178L_90d: 57227 rows
  CFDDataFlatRoof178W_0d: 57219 rows
  CFDDataFlatRoof178W_45d: 57148 rows
  CFDDataFlatRoof178W_90d: 57227 rows
  CFDDataFlatRoof89H_0d: 57046 rows
  CFDDataFlatRoof89H_45d: 57052 rows
  CFDDataFlatRoof89H_90d: 57071 rows
  CFDDataFlatRoof_0d: 42799 rows
  CFDDataFlatRoof_11.25d: 42812 rows
  CFDDataFlatRoof_22.5d: 42864 rows
  CFDDataFlatRoof_45d: 42817 rows
  CFDDataFlatRoof_90d: 42782 rows
  CFDDataGableRoof15d_0d: 44144 rows
  CFDDataGableRoof15d_45d: 43432 rows
  CFDDataGableRoof15d_90d: 43627 rows
  CFDDataGableRoof178L5d_0d: 53028 rows
  CFDDataGableRoof178L5d_45d: 48125 rows
  CFDDataGableRoof178L5d_90d: 53461 rows
  CFDDataGableRoof178W5d_0d: 62007 rows
  CFDDataGableRoof178W5d_45d: 60773 row

In [9]:
# Quick check: recount the datasets in the final output
from pathlib import Path

output_csv = Path("ProcessedData") / "compiled_processed_features.csv"
if output_csv.exists():
    df_check = pd.read_csv(output_csv)
    print("Dataset breakdown in final output:")
    print(df_check.groupby('dataset_name').size())
    print(f"\nTotal: {len(df_check)} rows from {df_check['dataset_name'].nunique()} datasets")
else:
    print(f"Output file not found: {output_csv}")

Dataset breakdown in final output:
dataset_name
CFDDataDome05_0d            12541
CFDDataDome10_0d            49706
CFDDataDome16_0d           105526
CFDDataFlatRoof178L_0d      57219
CFDDataFlatRoof178L_45d     57148
                            ...  
CFDDataWaves10_45d          52999
CFDDataWaves10_90d          56866
CFDDataWaves16_0d          135606
CFDDataWaves16_45d         135605
CFDDataWaves16_90d         135664
Length: 94, dtype: int64

Total: 5705217 rows from 94 datasets


In [10]:
# Validation: dist_from_leading_edge_along_wind should span [0, 1] per surface with NO NaN
from pathlib import Path
import pandas as pd

output_csv = Path("ProcessedData") / "compiled_processed_features.csv"

if output_csv.exists():
    df_check = pd.read_csv(output_csv)
    
    print("\n" + "="*70)
    print("VALIDATION: dist_from_leading_edge_along_wind Feature")
    print("="*70)
    
    if "dist_from_leading_edge_along_wind" in df_check.columns:
        feature = df_check["dist_from_leading_edge_along_wind"]
        
        total_rows = len(df_check)
        nan_count = feature.isna().sum()
        valid_count = total_rows - nan_count
        
        print(f"\nGlobal Statistics:")
        print(f"  Total rows:     {total_rows:>8}")
        print(f"  Valid rows:     {valid_count:>8}")
        print(f"  NaN rows:       {nan_count:>8} ({100*nan_count/total_rows:>5.1f}%)")
        
        if valid_count > 0:
            print(f"\nValue Range (valid only):")
            print(f"  Min:            {feature.min():>8.3f}")
            print(f"  Max:            {feature.max():>8.3f}")
            print(f"  Mean:           {feature.mean():>8.3f}")
            print(f"  Std:            {feature.std():>8.3f}")
            print(f"  25th percentile:{feature.quantile(0.25):>8.3f}")
            print(f"  Median:         {feature.median():>8.3f}")
            print(f"  75th percentile:{feature.quantile(0.75):>8.3f}")
        
        print(f"\nPer-Dataset Validation:")
        for dataset in sorted(df_check["dataset_name"].unique()):
            df_ds = df_check[df_check["dataset_name"] == dataset]
            feature_ds = df_ds["dist_from_leading_edge_along_wind"]
            nan_pct = 100 * feature_ds.isna().sum() / len(df_ds)
            
            if feature_ds.notna().any():
                print(f"  {dataset:20s}: NaN={nan_pct:5.1f}%, min={feature_ds.min():6.3f}, max={feature_ds.max():6.3f}")
            else:
                print(f"  {dataset:20s}: ❌ ALL NaN")
        
        # Check if feature is properly normalized
        valid_vals = feature[feature.notna()]
        mostly_valid = (valid_count / total_rows) > 0.9
        mostly_01_range = ((valid_vals >= -0.1) & (valid_vals <= 1.1)).sum() / len(valid_vals) > 0.95 if len(valid_vals) > 0 else False
        
        print(f"\n✓ PASSED" if (mostly_valid and mostly_01_range) else f"\n⚠ WARNING")
        if mostly_valid:
            print(f"  ✓ Feature covers >90% of data (no systematic NaN)")
        else:
            print(f"  ❌ Feature has >10% NaN ({nan_pct:.1f}%)")
        
        if mostly_01_range:
            print(f"  ✓ Feature values in expected [0, 1] range")
        else:
            print(f"  ⚠ Some values outside [0, 1]")
    else:
        print("❌ Feature 'dist_from_leading_edge_along_wind' not found in output CSV")
    
    print("="*70)
else:
    print(f"❌ Output file not found: {output_csv}")



VALIDATION: dist_from_leading_edge_along_wind Feature

Global Statistics:
  Total rows:      5705217
  Valid rows:      5705217
  NaN rows:              0 (  0.0%)

Value Range (valid only):
  Min:              -1.000
  Max:               1.000
  Mean:              0.381
  Std:               0.302
  25th percentile:   0.100
  Median:            0.356
  75th percentile:   0.631

Per-Dataset Validation:
  CFDDataDome05_0d    : NaN=  0.0%, min=-0.250, max= 0.965
  CFDDataDome10_0d    : NaN=  0.0%, min=-0.280, max= 0.966
  CFDDataDome16_0d    : NaN=  0.0%, min=-0.271, max= 0.964
  CFDDataFlatRoof178L_0d: NaN=  0.0%, min=-0.000, max= 1.000
  CFDDataFlatRoof178L_45d: NaN=  0.0%, min=-0.142, max= 0.975
  CFDDataFlatRoof178L_90d: NaN=  0.0%, min=-0.932, max= 1.000
  CFDDataFlatRoof178W_0d: NaN=  0.0%, min=-1.000, max= 1.000
  CFDDataFlatRoof178W_45d: NaN=  0.0%, min=-0.181, max= 0.960
  CFDDataFlatRoof178W_90d: NaN=  0.0%, min=-0.198, max= 1.000
  CFDDataFlatRoof89H_0d: NaN=  0.0%, min=-1.000

In [11]:
# FINAL SUMMARY: Multi-dataset feature engineering complete with all features
from pathlib import Path
import pandas as pd

output_csv = Path("ProcessedData") / "compiled_processed_features.csv"

if output_csv.exists():
    df_final = pd.read_csv(output_csv)
    
    print("\n" + "="*70)
    print("MULTI-DATASET FEATURE ENGINEERING - FINAL REPORT")
    print("="*70)
    
    print(f"\n✓ Successfully compiled {df_final['dataset_name'].nunique()} datasets")
    print(f"✓ Total observations: {len(df_final):,}")
    print(f"✓ Total features: {len(df_final.columns) - 1} (excluding dataset_name)")
    print(f"✓ Output file: {output_csv}")
    print(f"✓ File size: {output_csv.stat().st_size / 1e6:.1f} MB")
    
    print(f"\nDataset Distribution:")
    dataset_counts = df_final['dataset_name'].value_counts().sort_index()
    for name, count in dataset_counts.items():
        pct = 100 * count / len(df_final)
        print(f"  {name:15s}: {count:>7,} rows ({pct:5.1f}%)")
    
    print(f"\nFlow-Path Aware Features Added:")
    flow_path_features = [
        "dist_from_leading_edge_along_wind",
        "dist_to_nearest_edge",
        "streamwise_surface_distance",
        "wind_incidence_angle_rad",
        "z_normalized",
    ]
    for feat in flow_path_features:
        if feat in df_final.columns:
            null_pct = 100 * df_final[feat].isna().sum() / len(df_final)
            print(f"  ✓ {feat:40s}: {null_pct:5.1f}% null")
    
    print(f"\nNeighborhood Aggregation Features Added:")
    neighborhood_features = [
        "neighbor_cp_mean",
        "neighbor_cp_std",
        "neighbor_z_variance",
        "upstream_separation_score",
    ]
    for feat in neighborhood_features:
        if feat in df_final.columns:
            null_pct = 100 * df_final[feat].isna().sum() / len(df_final)
            print(f"  ✓ {feat:40s}: {null_pct:5.1f}% null")
    
    print(f"\nAll Feature Names ({len(df_final.columns) - 1}):")
    feat_cols = [c for c in df_final.columns if c != 'dataset_name']
    for i, col in enumerate(feat_cols, 1):
        print(f"  {i:2d}. {col}")
    
    if 'Cp' in df_final.columns:
        print(f"\nTarget Variable (Cp):")
        print(f"  Mean:  {df_final['Cp'].mean():>10.4f}")
        print(f"  Std:   {df_final['Cp'].std():>10.4f}")
        print(f"  Min:   {df_final['Cp'].min():>10.4f}")
        print(f"  Max:   {df_final['Cp'].max():>10.4f}")
        print(f"  Range: {df_final['Cp'].max() - df_final['Cp'].min():>10.4f}")
    
    print(f"\n✓ Dataset ready for ML training/validation with spatial context")
    print("="*70)
else:
    print(f"❌ Output file not found: {output_csv}")


MULTI-DATASET FEATURE ENGINEERING - FINAL REPORT

✓ Successfully compiled 94 datasets
✓ Total observations: 5,705,217
✓ Total features: 47 (excluding dataset_name)
✓ Output file: ProcessedData\compiled_processed_features.csv
✓ File size: 3978.2 MB

Dataset Distribution:
  CFDDataDome05_0d:  12,541 rows (  0.2%)
  CFDDataDome10_0d:  49,706 rows (  0.9%)
  CFDDataDome16_0d: 105,526 rows (  1.8%)
  CFDDataFlatRoof178L_0d:  57,219 rows (  1.0%)
  CFDDataFlatRoof178L_45d:  57,148 rows (  1.0%)
  CFDDataFlatRoof178L_90d:  57,227 rows (  1.0%)
  CFDDataFlatRoof178W_0d:  57,219 rows (  1.0%)
  CFDDataFlatRoof178W_45d:  57,148 rows (  1.0%)
  CFDDataFlatRoof178W_90d:  57,227 rows (  1.0%)
  CFDDataFlatRoof89H_0d:  57,046 rows (  1.0%)
  CFDDataFlatRoof89H_45d:  57,052 rows (  1.0%)
  CFDDataFlatRoof89H_90d:  57,071 rows (  1.0%)
  CFDDataFlatRoof_0d:  42,799 rows (  0.8%)
  CFDDataFlatRoof_11.25d:  42,812 rows (  0.8%)
  CFDDataFlatRoof_22.5d:  42,864 rows (  0.8%)
  CFDDataFlatRoof_45d:  42,8